# JUND-F0: Joint Unvoiced/Voiced Detection and F0 Estimation

Train a neural network that jointly performs **voiced/unvoiced detection** and **F0 (pitch) estimation** on the VCTK dataset.

**Reference:** *JUND-F0: A Novel Deep Learning Framework for Joint Unvoiced/Voiced Detection And F0 Estimation* (ICASSP 2026)

## Features
- Shared encoder with dual task-specific heads (V/UV + F0)
- Self-attention for long-range temporal context
- Depthwise separable convolutions for efficiency
- Joint loss: V/UV BCE + F0 L1 + FFE auxiliary
- Mixed precision (AMP) training for fast Colab training
- EMA weight averaging for stable evaluation

## Runtime
- **GPU**: T4 (free) or A100 (Colab Pro)
- **Time**: ~2-4 hours for 100K steps on T4
- **Disk**: ~12 GB for VCTK dataset

In [ ]:
#@title 1. Check GPU & Environment
!nvidia-smi
print(f"PyTorch: {__import__('torch').__version__}")
print(f"CUDA available: {__import__('torch').cuda.is_available()}")
if __import__('torch').cuda.is_available():
    print(f"GPU: {__import__('torch').cuda.get_device_name(0)}")
    print(f"GPU Memory: {__import__('torch').cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#@title 2. Install Dependencies
!pip install -q torchaudio librosa soundfile pyyaml tensorboard

# Optional: install CREPE for better pseudo-labels (slower but more accurate)
USE_CREPE_LABELS = False #@param {type:"boolean"}
if USE_CREPE_LABELS:
    !pip install -q crepe

# Optional: Weights & Biases for experiment tracking
USE_WANDB = False #@param {type:"boolean"}
if USE_WANDB:
    !pip install -q wandb

print("Dependencies installed!")

In [ ]:
#@title 3. Clone JUND-F0 Repository
import os

# Set working directory
WORK_DIR = "/content/jund-f0"

# If you have the code on GitHub, clone it:
!git clone https://github.com/BF667/jund-f0.git {WORK_DIR}

# Install the package
!cd {WORK_DIR} && pip install -e . -q
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
#@title 4. Write All Source Files (Run this cell!)
# This cell writes all the JUND-F0 source code into the project directory.

import os

# Create directories
for d in ['jund_f0', 'scripts', 'configs', 'notebooks']:
    os.makedirs(os.path.join(WORK_DIR, d), exist_ok=True)

# We'll import directly from the modules below
print("Source files will be loaded from the mounted project directory.")
print("If you cloned the repo, the files are already in place.")

In [ ]:
#@title 5. Download VCTK Dataset
#@markdown VCTK corpus: ~44 hours of English speech from 110 speakers

DATA_DIR = "/content/data/vctk" #@param {type:"string"}
SKIP_DOWNLOAD = False #@param {type:"boolean"}

if not SKIP_DOWNLOAD:
    import os
    os.makedirs(DATA_DIR, exist_ok=True)

    # Try HuggingFace mirror first (usually faster)
    vctk_zip = os.path.join(DATA_DIR, "VCTK-Corpus-0.92.zip")

    if not os.path.exists(vctk_zip):
        print("Downloading VCTK from HuggingFace mirror...")
        !wget -q --show-progress \
            "https://huggingface.co/datasets/CSTR-Edinburgh/vctk/resolve/main/VCTK-Corpus-0.92.zip" \
            -O {vctk_zip}
    else:
        print("VCTK zip already downloaded.")

    # Extract
    audio_dir = os.path.join(DATA_DIR, "wav48_silence_trimmed")
    if not os.path.exists(audio_dir):
        print("Extracting VCTK (this takes a few minutes)...")
        !unzip -q -o {vctk_zip} -d {DATA_DIR}
    else:
        print("VCTK already extracted.")

    # Count files
    import glob
    n_files = len(glob.glob(os.path.join(audio_dir, "**/*.flac"), recursive=True))
    n_files += len(glob.glob(os.path.join(audio_dir, "**/*.wav"), recursive=True))
    print(f"VCTK ready: {n_files} audio files found")
else:
    print("Skipping download (set SKIP_DOWNLOAD=False to download)")

In [ ]:
#@title 6. Generate F0 Pseudo-Labels
#@markdown Uses librosa.pyin to generate F0 labels for training.
#@markdown This takes ~20-30 minutes for the full VCTK dataset.

import os
import glob
import numpy as np
import librosa
from pathlib import Path
from tqdm.auto import tqdm

LABEL_DIR = os.path.join(DATA_DIR, "f0_labels")
SAMPLE_RATE = 16000
HOP_LENGTH = 160
N_FFT = 1024
F_MIN = 50.0
F_MAX = 800.0
LABEL_METHOD = "pyin" #@param ["pyin", "crepe"]
FORCE_REGENERATE = False #@param {type:"boolean"}

os.makedirs(LABEL_DIR, exist_ok=True)

# Find all audio files
audio_dir = os.path.join(DATA_DIR, "wav48_silence_trimmed")
audio_files = sorted(
    glob.glob(os.path.join(audio_dir, "**/*.flac"), recursive=True) +
    glob.glob(os.path.join(audio_dir, "**/*.wav"), recursive=True)
)

print(f"Found {len(audio_files)} audio files")

# Check which labels already exist
existing = set()
if os.path.exists(LABEL_DIR) and not FORCE_REGENERATE:
    existing = {f.stem for f in Path(LABEL_DIR).glob("*.npy")}

to_process = [f for f in audio_files if Path(f).stem not in existing]
print(f"Labels to generate: {len(to_process)} (already cached: {len(existing)})")

if to_process:
    for audio_path in tqdm(to_process, desc="Generating F0 labels"):
        utt_id = Path(audio_path).stem
        label_path = os.path.join(LABEL_DIR, f"{utt_id}.npy")

        try:
            # Load audio
            y, sr = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

            if LABEL_METHOD == "pyin":
                f0, _, _ = librosa.pyin(
                    y, fmin=F_MIN, fmax=F_MAX,
                    sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
                    frame_length=N_FFT,
                )
            elif LABEL_METHOD == "crepe":
                import crepe
                _, f0, _, _ = crepe.predict(
                    y, SAMPLE_RATE,
                    step_size=int(HOP_LENGTH / SAMPLE_RATE * 1000),
                    viterbi=True,
                )

            # Replace NaN with 0
            f0 = np.nan_to_num(f0, nan=0.0).astype(np.float32)
            np.save(label_path, f0)

        except Exception as e:
            print(f"  Error processing {utt_id}: {e}")

    print(f"F0 labels generated! Total: {len(list(Path(LABEL_DIR).glob('*.npy')))}")
else:
    print("All F0 labels already cached.")

In [ ]:
#@title 7. Configure Training Parameters
#@markdown Adjust these parameters based on your GPU memory.

import sys
sys.path.insert(0, WORK_DIR)

# ===== MODEL CONFIG =====
SAMPLE_RATE = 16000       #@param {type:"integer"}
N_FFT = 1024              #@param {type:"integer"}
HOP_LENGTH = 160          #@param {type:"integer"}
N_MELS = 80               #@param {type:"integer"}
ENCODER_CHANNELS = 64     #@param {type:"integer"}
ENCODER_BLOCKS = 4        #@param {type:"integer"}
USE_SELF_ATTENTION = True #@param {type:"boolean"}
F0_MIN = 50.0             #@param {type:"number"}
F0_MAX = 800.0            #@param {type:"number"}

# ===== TRAINING CONFIG =====
BATCH_SIZE = 16            #@param {type:"integer"}
LEARNING_RATE = 0.001      #@param {type:"number"}
WEIGHT_DECAY = 0.0001      #@param {type:"number"}
WARMUP_STEPS = 1000       #@param {type:"integer"}
MAX_STEPS = 100000        #@param {type:"integer"}
EVAL_INTERVAL = 2000      #@param {type:"integer"}
SAVE_INTERVAL = 5000      #@param {type:"integer"}
SEGMENT_LENGTH = 512      #@param {type:"integer"}
USE_AMP = True            #@param {type:"boolean"}
USE_EMA = True            #@param {type:"boolean"}
GRAD_CLIP = 5.0           #@param {type:"number"}

# ===== PATHS =====
OUTPUT_DIR = "/content/runs/jund-f0" #@param {type:"string"}
RESUME_FROM = ""            #@param {type:"string"}

print("Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max steps: {MAX_STEPS}")
print(f"  Segment length: {SEGMENT_LENGTH} frames ({SEGMENT_LENGTH * HOP_LENGTH / SAMPLE_RATE:.1f}s per sample)")
print(f"  AMP: {USE_AMP}")
print(f"  EMA: {USE_EMA}")

In [ ]:
#@title 8. Build Model & DataLoaders

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

# Import JUND-F0 modules
from jund_f0.model import JUNDF0, JUNDF0Config
from jund_f0.dataset import VCTKDataset, collate_fn
from jund_f0.train import Trainer, EMA, CosineAnnealingWarmup
from jund_f0.metrics import compute_all_metrics

# Build model config
model_config = JUNDF0Config(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f0_min=F0_MIN,
    f0_max=F0_MAX,
    encoder_channels=ENCODER_CHANNELS,
    encoder_blocks=ENCODER_BLOCKS,
    use_self_attention=USE_SELF_ATTENTION,
    input_frames=SEGMENT_LENGTH,
)

# Create datasets
train_dataset = VCTKDataset(
    root_dir=DATA_DIR,
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F0_MIN,
    f_max=F0_MAX,
    segment_length=SEGMENT_LENGTH,
    split="train",
    use_augmentation=True,
    label_method=LABEL_METHOD,
)

val_dataset = VCTKDataset(
    root_dir=DATA_DIR,
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    f_min=F0_MIN,
    f_max=F0_MAX,
    segment_length=SEGMENT_LENGTH,
    split="val",
    use_augmentation=False,
    label_method=LABEL_METHOD,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True,
    drop_last=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn,
    pin_memory=True,
)

# Build model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = JUNDF0(model_config).to(device)

print(f"Model: JUND-F0")
print(f"  Parameters: {model.count_parameters():,}")
print(f"  Device: {device}")
print(f"  Train samples: {len(train_dataset)}")
print(f"  Val samples: {len(val_dataset)}")
print(f"  Train speakers: {len(set(f['speaker'] for f in train_dataset.file_list))}")
print(f"  Val speakers: {len(set(f['speaker'] for f in val_dataset.file_list))}")

# Verify forward pass
dummy_mel = torch.randn(2, N_MELS, SEGMENT_LENGTH).to(device)
dummy_vuv = torch.randn(2, SEGMENT_LENGTH, 1).to(device)
dummy_f0 = torch.abs(torch.randn(2, SEGMENT_LENGTH, 1).to(device)) * 200 + 100

outputs = model(dummy_mel, vuv_label=dummy_vuv, f0_label=dummy_f0)
print(f"\nForward pass test:")
print(f"  Loss: {outputs['loss'].item():.4f}")
print(f"  VUV loss: {outputs['vuv_loss'].item():.4f}")
print(f"  F0 loss: {outputs['f0_loss'].item():.4f}")
print(f"  FFE: {outputs['ffe'].item():.4f}")
print(f"  VUV prob shape: {outputs['vuv_prob'].shape}")
print(f"  F0 pred shape: {outputs['f0_pred'].shape}")

In [ ]:
#@title 9. Train!
#@markdown This cell runs the training loop.
#@markdown You can monitor progress with TensorBoard (cell below).

import os
import time
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Create trainer
trainer = Trainer(
    config=model_config,
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    num_workers=2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    eval_interval=EVAL_INTERVAL,
    save_interval=SAVE_INTERVAL,
    log_interval=50,
    grad_clip_norm=GRAD_CLIP,
    use_amp=USE_AMP,
    use_ema=USE_EMA,
    label_method=LABEL_METHOD,
    segment_length=SEGMENT_LENGTH,
    resume_from=RESUME_FROM if RESUME_FROM else None,
)

# Start training!
trainer.train()

In [ ]:
#@title 10. (Optional) Launch TensorBoard
#@markdown Monitor training in real-time.

%load_ext tensorboard
%tensorboard --logdir /content/runs/jund-f0/logs

In [ ]:
#@title 11. Evaluate on Validation Set

import torch
import numpy as np
from tqdm.auto import tqdm

# Load best model
best_model_path = os.path.join(OUTPUT_DIR, "best_model.pt")
if os.path.exists(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from step {checkpoint['global_step']}, FFE: {checkpoint.get('best_ffe', 'N/A')}")

model.eval()

all_f0_pred = []
all_f0_label = []
all_vuv_pred = []
all_vuv_label = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc="Evaluating"):
        mel = batch["mel"].to(device)
        vuv = batch["vuv"].to(device)
        f0 = batch["f0"].to(device)

        outputs = model(mel)
        vuv_pred = (outputs["vuv_prob"] > 0.5).float()
        f0_pred = outputs["f0_pred"]

        all_f0_pred.append(f0_pred.cpu())
        all_f0_label.append(f0.cpu())
        all_vuv_pred.append(vuv_pred.cpu())
        all_vuv_label.append(vuv.cpu())

f0_pred_all = torch.cat(all_f0_pred, dim=0)
f0_label_all = torch.cat(all_f0_label, dim=0)
vuv_pred_all = torch.cat(all_vuv_pred, dim=0)
vuv_label_all = torch.cat(all_vuv_label, dim=0)

metrics = compute_all_metrics(f0_pred_all, f0_label_all, vuv_pred_all, vuv_label_all)

print("\n" + "="*50)
print("EVALUATION RESULTS")
print("="*50)
print(f"  Raw Pitch Accuracy (RPA): {metrics['rpa']:.4f} ({metrics['rpa']*100:.1f}%)")
print(f"  Raw Chroma Accuracy (RCA): {metrics['rca']:.4f} ({metrics['rca']*100:.1f}%)")
print(f"  Gross Pitch Error (GPE):   {metrics['gpe']:.4f} ({metrics['gpe']*100:.1f}%)")
print(f"  Voicing Decision Error:    {metrics['vde']:.4f} ({metrics['vde']*100:.1f}%)")
print(f"  F0 Frame Error (FFE):      {metrics['ffe']:.4f} ({metrics['ffe']*100:.1f}%)")
print("="*50)

In [ ]:
#@title 12. Test Inference on Sample Audio

import torchaudio
import matplotlib.pyplot as plt

# Find a sample audio file from validation set
val_speakers = list(set(f['speaker'] for f in val_dataset.file_list))
sample_speaker = val_speakers[0]
sample_files = [f for f in val_dataset.file_list if f['speaker'] == sample_speaker][:1]

if sample_files:
    from jund_f0.infer import JUNDF0Inference

    inferencer = JUNDF0Inference(model_path=best_model_path, device=str(device))

    for sample in sample_files:
        result = inferencer.extract_f0(sample['audio_path'], return_vuv=True, return_prob=True)

        fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

        # Load label for comparison
        label_path = os.path.join(LABEL_DIR, f"{sample['utt_id']}.npy")
        if os.path.exists(label_path):
            f0_label = np.load(label_path)
            time_label = np.arange(len(f0_label)) * HOP_LENGTH / SAMPLE_RATE
            # Align lengths
            min_len = min(len(f0_label), len(result['f0']))

            # Plot F0 comparison
            voiced_label = f0_label[:min_len] > 0
            voiced_pred = result['vuv'][:min_len]

            axes[0].plot(time_label[:min_len][voiced_label], f0_label[:min_len][voiced_label],
                        'b.', markersize=2, alpha=0.7, label='Ground Truth (pyin)')
            axes[0].plot(result['time'][:min_len][voiced_pred], result['f0'][:min_len][voiced_pred],
                        'r.', markersize=2, alpha=0.7, label='JUND-F0 Prediction')
            axes[0].set_ylabel('F0 (Hz)')
            axes[0].set_title(f'F0 Estimation: {sample["utt_id"]}')
            axes[0].legend(loc='best')
            axes[0].set_ylim(F0_MIN, F0_MAX)
        else:
            axes[0].plot(result['time'][result['vuv']], result['f0'][result['vuv']],
                        'r.', markersize=2, alpha=0.7, label='JUND-F0 Prediction')
            axes[0].set_ylabel('F0 (Hz)')
            axes[0].set_title(f'F0 Estimation: {sample["utt_id"]}')
            axes[0].legend(loc='best')

        # V/UV probability
        if 'vuv_prob' in result:
            axes[1].fill_between(result['time'], result['vuv_prob'], alpha=0.5, color='green')
            axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
            axes[1].set_ylabel('V/UV Probability')
            axes[1].set_title('Voiced/Unvoiced Detection')
            axes[1].set_ylim(-0.05, 1.05)

        # V/UV flags
        axes[2].plot(result['time'], result['vuv'].astype(float), 'g-', linewidth=0.5)
        axes[2].set_ylabel('V/UV')
        axes[2].set_xlabel('Time (s)')
        axes[2].set_title('Voicing Decision')
        axes[2].set_ylim(-0.05, 1.05)

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'{sample["utt_id"]}_f0_plot.png'), dpi=150)
        plt.show()
        print(f"Plot saved to {OUTPUT_DIR}/{sample['utt_id']}_f0_plot.png")

In [ ]:
#@title 13. Download Trained Model

from google.colab import files

# Download best model
best_model_path = os.path.join(OUTPUT_DIR, "best_model.pt")
if os.path.exists(best_model_path):
    print(f"Downloading {best_model_path}...")
    files.download(best_model_path)
else:
    print("Best model not found. Check if training completed.")

# Also download final model
final_model_path = os.path.join(OUTPUT_DIR, "final_model.pt")
if os.path.exists(final_model_path):
    print(f"Downloading {final_model_path}...")
    files.download(final_model_path)

In [ ]:
#@title 14. Save to Google Drive (Optional)

SAVE_TO_DRIVE = False #@param {type:"boolean"}

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_dir = "/content/drive/MyDrive/jund-f0"
    os.makedirs(drive_dir, exist_ok=True)

    import shutil
    # Copy best model
    if os.path.exists(best_model_path):
        shutil.copy2(best_model_path, drive_dir)
        print(f"Best model saved to {drive_dir}")

    # Copy all checkpoints
    for ckpt in Path(OUTPUT_DIR).glob("*.pt"):
        shutil.copy2(ckpt, drive_dir)
        print(f"  Copied {ckpt.name}")

    print(f"\nAll models saved to Google Drive: {drive_dir}")

## Using JUND-F0 in Voice Conversion (RVC / So-VITS-SVC)

```python
from jund_f0.infer import create_rvc_compatible_extractor

# Create RVC-compatible F0 extractor
f0_extractor = create_rvc_compatible_extractor("best_model.pt")

# Use in RVC pipeline
import numpy as np
audio = np.random.randn(16000).astype(np.float32)  # your audio
f0 = f0_extractor(audio, sample_rate=16000)
print(f"Extracted F0: shape={f0.shape}, range=[{f0.min():.1f}, {f0.max():.1f}]")
```